#### This Notebook deals with data accessing and cleaning before performing EDA

In [1]:
import pandas as pd
import numpy as np

In [2]:
patients = pd.read_csv('https://raw.githubusercontent.com/campusx-official/data-wrangling/master/patients.csv')
treatments = pd.read_csv('https://raw.githubusercontent.com/campusx-official/data-wrangling/master/treatments.csv')
adverse_reactions = pd.read_csv('https://raw.githubusercontent.com/campusx-official/data-wrangling/master/adverse_reactions.csv')
treatments_cut = pd.read_csv('https://raw.githubusercontent.com/campusx-official/data-wrangling/master/treatments_cut.csv')

In [3]:
patients.head()

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
0,1,female,Zoe,Wellish,576 Brown Bear Drive,Rancho California,California,92390.0,United States,951-719-9170ZoeWellish@superrito.com,7/10/1976,121.7,66,19.6
1,2,female,Pamela,Hill,2370 University Hill Road,Armstrong,Illinois,61812.0,United States,PamelaSHill@cuvox.de+1 (217) 569-3204,4/3/1967,118.8,66,19.2
2,3,male,Jae,Debord,1493 Poling Farm Road,York,Nebraska,68467.0,United States,402-363-6804JaeMDebord@gustr.com,2/19/1980,177.8,71,24.8
3,4,male,Liêm,Phan,2335 Webster Street,Woodbridge,NJ,7095.0,United States,PhanBaLiem@jourrapide.com+1 (732) 636-8246,7/26/1951,220.9,70,31.7
4,5,male,Tim,Neudorf,1428 Turkey Pen Lane,Dothan,AL,36303.0,United States,334-515-7487TimNeudorf@cuvox.de,2/18/1928,192.3,27,26.1


In [4]:
# export data for manual assessment

with pd.ExcelWriter('clinical_trials.xlsx') as writer:
  patients.to_excel(writer,sheet_name='patients')
  treatments.to_excel(writer,sheet_name='treatments')
  treatments_cut.to_excel(writer,sheet_name='treatments_cut')
  adverse_reactions.to_excel(writer,sheet_name='adverse_reactions')

#### Details of the Data
Table Details:

(i) `Patient Table:`  Total 503 entries with 12 nulls in a few columns.

This table has the details of all the patients, patient id, their gender, first name, surname, address they live at, city, state, zip code, country, contact, date of birth, weight, height and BMI.

(ii) `Treatment and Treatment_cut Table:` Total 280 entries with around 100 mulls in hba1c_change column. Treatment_cut has 70 entries/

This table gives us the details of the treatments given to the patients.

The columns include first name, surname, auralin and novodra, hba1c start, hba1c end and hba1c change column are also given.

(iii) `Adverse reaction Table:` 34 entries with NO NULLS.

This table shows the patients who got reactions due to the ongoing treatment.

Contains given name, surname of the patients and the reaction they suffered from.



### Problems with the data

`Dirty Data:`

#### Patients Table: Every category column must be lower case and stripped
`(i) patient_id 9`  -> given name -> Dsivd instead of David  `accuracy`
`(ii) missing details` address, city, state and contact Index 209, 219, 230, 234, 242, 249, 257, 264, 269, 278, 286, 296 `completeness`
`(iii) State column` Some state in abbreviations and some are full names.   `consistency`
`(iv) zip codes ->` there are a few zips with 4 digits only.  `accuracy`
`(v) Index 210 ->` weight is very less (check it once). Can use bmi to check if it is correct. `accuracy`
`(vi) Unknown Data ->` patient id 230,238,245,252,278 Unknown data named by John Doe `validity`  

#### Treatments and cut table: Every category column must be lower case and stripped

`(i) given name`, surname are in lower cases. `consistency`
`(ii) Column Auralin and Novodra ->` Instead of NULLS (-) is given. `validity`
`(iii) hba1c change` is calculated wrongly. and their are missing values also. `validity`
`(iv) given name -> joseph surname -> day` duplicate row must be removed. `validity`


`Messy Data:`

#### Patients Table:
`(i) Contact column ->` mail and number together -> separate them in two columns.

#### Treatments and cut table:

`(i) auralin and novodra colum`n contains dosage start and end values which we should separate in two columns auralin_start, auralin_end and same for novodra. Also remove unit from them. (Make it int type)

`(ii) Merge` the treatment and cut tables. 

#### adverse reactions:
`(ii) We don't need this table,` just join the reaction column in patient table with the patient given and surname. 

In [5]:
patients_df = patients.copy()
treatments_df = treatments.copy()
treatments_cut_df = treatments_cut.copy()
adverse_reactions_df = adverse_reactions.copy()

In [6]:
patients_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   patient_id    503 non-null    int64  
 1   assigned_sex  503 non-null    object 
 2   given_name    503 non-null    object 
 3   surname       503 non-null    object 
 4   address       491 non-null    object 
 5   city          491 non-null    object 
 6   state         491 non-null    object 
 7   zip_code      491 non-null    float64
 8   country       491 non-null    object 
 9   contact       491 non-null    object 
 10  birthdate     503 non-null    object 
 11  weight        503 non-null    float64
 12  height        503 non-null    int64  
 13  bmi           503 non-null    float64
dtypes: float64(3), int64(2), object(9)
memory usage: 55.1+ KB


In [8]:
treatments_cut_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   given_name    70 non-null     object 
 1   surname       70 non-null     object 
 2   auralin       70 non-null     object 
 3   novodra       70 non-null     object 
 4   hba1c_start   70 non-null     float64
 5   hba1c_end     70 non-null     float64
 6   hba1c_change  42 non-null     float64
dtypes: float64(3), object(4)
memory usage: 4.0+ KB


In [9]:
adverse_reactions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   given_name        34 non-null     object
 1   surname           34 non-null     object
 2   adverse_reaction  34 non-null     object
dtypes: object(3)
memory usage: 948.0+ bytes


#### Automatic Accessment

In [10]:
treatments_df.head()

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change
0,veronika,jindrová,41u - 48u,-,7.63,7.20,NaN
1,elliot,richardson,-,40u - 45u,7.56,7.09,0.97
2,yukitaka,takenaka,-,39u - 36u,7.68,7.25,NaN
3,skye,gormanston,33u - 36u,-,7.97,7.62,0.35
4,alissa,montez,-,33u - 29u,7.78,7.46,0.32


In [ ]:
treatments[treatments.duplicated()] # Duplicate row in treatment table

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change
136,joseph,day,29u - 36u,-,7.7,7.19,NaN


#### Patients Table

In [ ]:
patients[patients[['given_name','surname']].duplicated()] # patient id 230,238,245,252,278 Unknown data 

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
229,230,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,johndoe@email.com1234567890,1/1/1975,180.0,72,24.4
237,238,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,johndoe@email.com1234567890,1/1/1975,180.0,72,24.4
244,245,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,johndoe@email.com1234567890,1/1/1975,180.0,72,24.4
251,252,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,johndoe@email.com1234567890,1/1/1975,180.0,72,24.4
277,278,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,johndoe@email.com1234567890,1/1/1975,180.0,72,24.4


In [20]:
patients['city'].value_counts()

city
New York        18
San Diego        8
Tulsa            7
Chicago          6
Houston          6
                ..
Fresno           1
Florence         1
South Boston     1
Natchez          1
Rudyard          1
Name: count, Length: 349, dtype: int64

In [21]:
patients_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   patient_id    503 non-null    int64  
 1   assigned_sex  503 non-null    object 
 2   given_name    503 non-null    object 
 3   surname       503 non-null    object 
 4   address       491 non-null    object 
 5   city          491 non-null    object 
 6   state         491 non-null    object 
 7   zip_code      491 non-null    float64
 8   country       491 non-null    object 
 9   contact       491 non-null    object 
 10  birthdate     503 non-null    object 
 11  weight        503 non-null    float64
 12  height        503 non-null    int64  
 13  bmi           503 non-null    float64
dtypes: float64(3), int64(2), object(9)
memory usage: 55.1+ KB


In [27]:
# patients table category columns lower case and changing the dtype of necessary columns
patients_df['assigned_sex'] = patients_df['assigned_sex'].str.lower().str.strip().astype('category')
patients_df['given_name'] = patients_df['given_name'].str.lower().str.strip()
patients_df['surname'] =  patients_df['surname'].str.lower().str.strip()
patients_df['address'] =  patients_df['address'].str.lower().str.strip()
patients_df['city'] =  patients_df['city'].str.lower().str.strip()
patients_df['state'] =  patients_df['state'].str.lower().str.strip()
patients_df['birthdate'] = pd.to_datetime(patients_df['birthdate'])


In [28]:
patients_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   patient_id    503 non-null    int64         
 1   assigned_sex  503 non-null    category      
 2   given_name    503 non-null    object        
 3   surname       503 non-null    object        
 4   address       491 non-null    object        
 5   city          491 non-null    object        
 6   state         491 non-null    object        
 7   zip_code      491 non-null    float64       
 8   country       491 non-null    object        
 9   contact       491 non-null    object        
 10  birthdate     503 non-null    datetime64[ns]
 11  weight        503 non-null    float64       
 12  height        503 non-null    int64         
 13  bmi           503 non-null    float64       
dtypes: category(1), datetime64[ns](1), float64(3), int64(2), object(7)
memory usage: 51.8+ KB


In [39]:
patients_df[patients_df['bmi'].isna()]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi


In [40]:
treatments_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   given_name    280 non-null    object 
 1   surname       280 non-null    object 
 2   auralin       280 non-null    object 
 3   novodra       280 non-null    object 
 4   hba1c_start   280 non-null    float64
 5   hba1c_end     280 non-null    float64
 6   hba1c_change  171 non-null    float64
dtypes: float64(3), object(4)
memory usage: 15.4+ KB


### Cleaning the Data

In [ ]:
patients_df.drop()

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
229,230,male,john,doe,123 main street,new york,ny,12345.0,United States,johndoe@email.com1234567890,1975-01-01,180.0,72,24.4
237,238,male,john,doe,123 main street,new york,ny,12345.0,United States,johndoe@email.com1234567890,1975-01-01,180.0,72,24.4
244,245,male,john,doe,123 main street,new york,ny,12345.0,United States,johndoe@email.com1234567890,1975-01-01,180.0,72,24.4
251,252,male,john,doe,123 main street,new york,ny,12345.0,United States,johndoe@email.com1234567890,1975-01-01,180.0,72,24.4
277,278,male,john,doe,123 main street,new york,ny,12345.0,United States,johndoe@email.com1234567890,1975-01-01,180.0,72,24.4
